# 19 — Conv-NN Residual Augmenter

The 15-coefficient harmonic basis + empirical residual map (NB 15) capture nearly everything a real detector throws at us. For the last few µε of pathological cases — irregular panel boundaries, fingerprint-shaped fabrication artefacts, large parallax in thick-sensor systems — `midas_calibrate_v2.pipelines.nn_residual.autocalibrate_nn` trains a small convolutional network to learn an extra ΔR(Y, Z) field that absorbs whatever the analytical model + the smooth empirical map left over.

Two modes:

* **`two_stage`** (default) — seed the geometry with the analytical engine, freeze it, then train the NN.
* **`joint`** — refine geometry and NN weights together (more powerful, easier to over-fit; not the default for a reason).

This notebook:

1. Walks through the API.
2. Shows the *per-harmonic drift report* — the diagnostic that proves the NN didn't absorb physical signal that should still live in `p_k`.
3. Discusses when **not** to use this (95% of cases).

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import h5py, numpy as np, torch
from pathlib import Path

## 1. The API

In [ ]:
from midas_calibrate_v2.pipelines.nn_residual import (
    autocalibrate_nn, NNCalibrationResult,
)
from midas_calibrate_v2.forward.nn_residual import NNResidualConfig

# The conv-net config — defaults are reasonable; tweak hidden_channels and
# kernel_size to trade fit quality for compute.
help(NNResidualConfig)

## 2. Train (two-stage mode)

First the alternating engine converges the analytical model; then a `ResidualConvNet` is fitted on top via Adam with smoothness + weight-decay regularizers.

In [ ]:
DATA = Path(os.environ.get('V2_ONESHOT_FILE',
    '/Users/hsharma/Desktop/analysis/leighanne/data/CeO2_XRD_echem_JLi_002587.vrx.h5'))
if not DATA.exists():
    raise FileNotFoundError(f'set V2_ONESHOT_FILE; missing {DATA}')
with h5py.File(DATA, 'r') as f:
    img  = f['exchange/data'][0].astype(np.float32)
    dark = f['exchange/data_dark'][0].astype(np.float32)

# Build v1 params from the leighanne setup.
from midas_calibrate.params import CalibrationParams
v1 = CalibrationParams(
    NrPixelsY=2880, NrPixelsZ=2880, pxY=150.0, pxZ=150.0,
    Lsd=840_000.0, BC_y=1430.0, BC_z=1472.0,
    tx=0.0, ty=0.0, tz=0.0,
    Wavelength=0.184139, SpaceGroup=225,
    LatticeConstant=(5.4116, 5.4116, 5.4116, 90.0, 90.0, 90.0),
    MaxRingRad=2000.0, MinRingRad=120.0, nIterations=4,
    Refine={'Lsd':True,'BC':True,'ty':True,'tz':True,
             'Wavelength':False,'Parallax':False,
             **{f'p{i}':True for i in range(15)}},
    Device='cpu', Dtype='fp64',
)

# Two-stage NN training
result = autocalibrate_nn(
    v1, img, dark=dark,
    mode='two_stage',
    n_iter_seed=3, n_steps_nn=500,
    nn_lr=1e-3, weight_decay_coef=1e-4, smoothness_coef=1e-3,
    device='cpu', verbose=True,
)
print('\nFinal MAP (geometry):')
for k in ('Lsd','BC_y','BC_z','ty','tz'):
    print(f'  {k:5s} = {float(result.map_unpacked[k]):+.4f}')
print(f'\nFinal NN loss: {result.losses[-1]:.3e}')
print(f'NN training curve: {result.losses[0]:.3e} -> {result.losses[-1]:.3e}'
      f' ({len(result.losses)} steps)')

## 3. The per-harmonic drift report (anti-cheating diagnostic)

Critical: a powerful NN can swallow physical signal that should live in the analytical p_k coefficients. `result.harmonic_drift` is the **change in each harmonic coefficient between the pre-NN (seed) MAP and the post-NN MAP**. If the NN is doing its job — capturing residuals the analytical model can't represent — the drift should be **small** (< 10% of the pre-NN values). Large drift means the NN is absorbing physical signal.

In [ ]:
print('Per-harmonic drift between pre-NN and post-NN MAP:')
print(f'  {"param":<10s} {"drift":>14s}')
for k, v in sorted(result.harmonic_drift.items()):
    flag = '  ←  >10% drift, suspicious' if abs(v) > 0.1 else ''
    print(f'  {k:<10s} {v:+14.4e}{flag}')

## 4. Visualise the learned ΔR(Y, Z) field

Evaluate the trained network on the full detector grid.

In [ ]:
import matplotlib.pyplot as plt
Y, Z = torch.meshgrid(
    torch.arange(2880, dtype=torch.float64),
    torch.arange(2880, dtype=torch.float64),
    indexing='xy',
)
with torch.no_grad():
    nn_dR = result.nn_model(Y, Z).numpy()

vlim = float(np.percentile(np.abs(nn_dR), 99))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(nn_dR, origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
ax.plot(float(result.map_unpacked['BC_y']),
        float(result.map_unpacked['BC_z']), 'ko', ms=6)
ax.set_title(f'Conv-NN residual ΔR(Y, Z) in pixels\n'
             f'rms = {float(np.sqrt((nn_dR**2).mean())):.4f} px')
plt.colorbar(im, ax=ax, label='ΔR (px)')
plt.tight_layout(); plt.show()

## When NOT to use the NN augmenter

* **You haven't tried the empirical residual map first (NB 15).** That handles 95% of what the NN would learn — at zero training cost and far easier to interpret.
* **Sparse rings / small calibrant data.** The NN needs enough fits across the detector to avoid overfitting to per-ring sampling holes.
* **You don't have a way to validate.** A second-calibrant cross-check (Si after CeO₂) or an independent strain test on a known-strain sample is the only way to confirm the NN didn't memorize.

**Good** signal you needed it: per-harmonic drift < 10% across all p_k AND post-NN strain visibly lower than the residual-map-only post-strain on the same data.

**Bad** signal: harmonic drift on `iso_R2` of 50%+ — the NN is just stealing the radial isotropic distortion.